# Experiment 5: Model Training

**Dataset:** Heart Disease Cleveland Dataset

### Objective
Train and compare Logistic Regression, Random Forest, and SVM models using 5-fold stratified cross-validation. Save the trained models for evaluation in Experiment 6.

## Step 1: Import Required Libraries

We use simple and commonly used Scikit-learn tools for preprocessing, model training, cross-validation, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


## Step 2: Load the Dataset

In [ ]:
df = pd.read_csv("Heart_disease_cleveland_new.csv")

print("Dataset shape:", df.shape)
df.head()


## Step 3: Separate Features and Target

`target` is the output variable:
- 0 = No heart disease
- 1 = Heart disease present

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

print("Features:", X.shape)
print("Target:", y.shape)


## Step 4: Train-Test Split

We keep 20% of the data untouched for Experiment 6. Stratification keeps the class proportion similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Test data:", X_test.shape)


## Step 5: Define Numerical and Categorical Features

These are the same feature groups used in Experiment 3.

In [ ]:
numerical_features = [
    "age", "trestbps", "chol", "thalach", "oldpeak"
]

categorical_features = [
    "sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"
]

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


## Step 6: Create Preprocessing Pipeline

Preprocessing is included inside the model pipeline. This is important because cross-validation must learn preprocessing only from each training fold, preventing data leakage.

In [ ]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])


## Step 7: Define the Three Candidate Models

We train three different classification algorithms as required by the lab manual.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42)
}

print("Models:", list(models.keys()))


## Step 8: 5-Fold Stratified Cross-Validation

Because the dataset is small, we use 5-fold stratified cross-validation on the training data. Recall is included because Experiment 1 identified disease-positive recall as the primary metric.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

results = []
trained_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    
    results.append({
        "Model": name,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Precision Mean": scores["test_precision"].mean(),
        "Recall Mean": scores["test_recall"].mean(),
        "F1 Mean": scores["test_f1"].mean(),
        "Recall Std": scores["test_recall"].std()
    })
    
    # Fit the complete training data after cross-validation
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

results_df = pd.DataFrame(results)
results_df


## Step 9: Compare Cross-Validation Results

The table below shows the average performance across the 5 validation folds. Higher recall means the model identifies more disease-positive cases.

In [ ]:
results_df = results_df.sort_values("Recall Mean", ascending=False)
results_df.round(3)


## Step 10: Display Training Accuracy

This gives the performance of each fitted model on the training data. It is useful for checking whether a model may be overfitting, but the cross-validation results are more useful for comparing generalization during this experiment.

In [ ]:
training_results = []

for name, pipeline in trained_pipelines.items():
    train_accuracy = pipeline.score(X_train, y_train)
    training_results.append({
        "Model": name,
        "Training Accuracy": train_accuracy
    })

training_df = pd.DataFrame(training_results)
training_df.round(3)


## Step 11: Save the Trained Models

The complete pipelines are saved. Each saved artifact contains both preprocessing and the trained model, so the same transformations can be applied later in Experiment 6.

In [ ]:
joblib.dump(trained_pipelines["Logistic Regression"], "logistic_regression.pkl")
joblib.dump(trained_pipelines["Random Forest"], "random_forest.pkl")
joblib.dump(trained_pipelines["SVM"], "svm.pkl")

print("Models saved successfully.")
print("1. logistic_regression.pkl")
print("2. random_forest.pkl")
print("3. svm.pkl")


## Step 12: Experiment 5 Conclusion

Three classification models were trained: Logistic Regression, Random Forest, and SVM. Stratified 5-fold cross-validation was used on the training data to compare accuracy, precision, recall, and F1-score. The 20% test set was kept separate for Experiment 6. The trained model pipelines were saved for final evaluation.

**Important:** The final model should not be selected using the test set in this experiment. Experiment 6 will evaluate the saved models on the untouched test data.